In [2]:
import sqlite3
conn = sqlite3.connect('ravenstack.db')

In [3]:
cursor = conn.cursor()

In [4]:
import pandas as pd
df_accounts = pd.read_sql_query("SELECT * FROM accounts;", conn)
df_subscriptions = pd.read_sql_query("SELECT * FROM subscriptions;", conn)
df_feature_usage = pd.read_sql_query("SELECT * FROM feature_usage;", conn)
df_support_tickets = pd.read_sql_query("SELECT * FROM support_tickets;", conn)
df_churn_events = pd.read_sql_query("SELECT * FROM churn_events;", conn)

In [5]:
total_accounts = df_accounts.shape[0] # .shape[0] means total rows

churned_accounts = df_accounts["churn_flag"].sum()

churn_rate = round((churned_accounts / total_accounts) * 100, 2)

print("Total Accounts:", total_accounts)
print("Churned Accounts:", churned_accounts)
print("Churn Rate:", churn_rate, "%")

Total Accounts: 500
Churned Accounts: 110
Churn Rate: 22.0 %


In [6]:
# Total accounts per industry
total_accounts = df_accounts.groupby("industry")["account_id"].count()

# Churned accounts per industry
churned_accounts = df_accounts.groupby("industry")["churn_flag"].sum()

# Combine both into one DataFrame
churn_by_industry = pd.DataFrame({
    "total_accounts": total_accounts,
    "churned_accounts": churned_accounts
})

# Calculate churn rate
churn_by_industry["churn_rate"] = round(
    (churn_by_industry["churned_accounts"] /
     churn_by_industry["total_accounts"]) * 100,
    2
)

# Sort by highest churn rate
churn_by_industry = churn_by_industry.sort_values(
    by="churn_rate",
    ascending=False
)

print(churn_by_industry)

               total_accounts  churned_accounts  churn_rate
industry                                                   
DevTools                  113                35       30.97
FinTech                   112                25       22.32
HealthTech                 96                21       21.88
EdTech                     79                13       16.46
Cybersecurity             100                16       16.00


In [7]:
# Merge accounts and support tickets
merged_df = df_accounts.merge(
    df_support_tickets,
    on="account_id",
    how="left"
)

# Count tickets per account
ticket_count = merged_df.groupby(
    ["account_id", "churn_flag"]
)["ticket_id"].count().reset_index()

# Rename column
ticket_count = ticket_count.rename(
    columns={"ticket_id": "ticket_count"}
)

# Average tickets by churn status
result = ticket_count.groupby(
    "churn_flag"
)["ticket_count"].mean()

print(result)

churn_flag
0    4.020513
1    3.927273
Name: ticket_count, dtype: float64


In [8]:
usage_summary = df_feature_usage.groupby("subscription_id")["usage_count"].sum().reset_index()

merged_df = usage_summary.merge(
    df_subscriptions,
    on="subscription_id",
    how="inner"
)

feature_usage = merged_df.groupby("churn_flag")["usage_count"].mean().reset_index()

feature_usage = feature_usage.rename(
    columns={"usage_count": "avg_feature_usage"}
)

print(feature_usage)

   churn_flag  avg_feature_usage
0           0          50.521186
1           1          49.244306


In [9]:
revenue_df = df_subscriptions[
    df_subscriptions["churn_flag"] == 1
].copy()

revenue_df["revenue_rank"] = revenue_df.groupby(
    "plan_tier"
)["mrr_amount"].rank(
    method="min",
    ascending=False
)

top_revenue = revenue_df[
    revenue_df["revenue_rank"] <= 3
]

top_revenue = top_revenue.sort_values(
    ["plan_tier", "revenue_rank"]
)

print(top_revenue)

     subscription_id account_id  start_date    end_date   plan_tier  seats  \
3900        S-51cb35   A-18793f  2024-12-20  2024-12-29       Basic    148   
3888        S-bd1912   A-0cc442  2024-10-06  2024-10-23       Basic     97   
1338        S-9dda85   A-462d45  2024-01-16  2024-05-29       Basic     89   
4280        S-9f8b23   A-118f1c  2024-12-24  2024-12-26  Enterprise    128   
184         S-7904d3   A-d4e0d4  2024-11-30  2024-12-03  Enterprise    117   
2155        S-79d1e0   A-ce550d  2024-12-31  2024-12-31  Enterprise    109   
4581        S-59f0a6   A-104f1a  2024-09-02  2024-11-28         Pro    123   
1668        S-5da6e7   A-5a215a  2024-07-29  2024-09-19         Pro     87   
2150        S-2223bf   A-5a215a  2024-10-23  2024-11-26         Pro     87   

      mrr_amount  arr_amount  is_trial  upgrade_flag  downgrade_flag  \
3900        2812       33744         0             0               0   
3888        1843       22116         0             0               0   
133

In [10]:
conn.commit()